In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
# importing for ANN
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense
from keras.layers import Dropout,Conv2D,MaxPooling2D,Flatten
#import kaggle



In [4]:
pwd

'/content'

In [6]:
df=pd.read_csv('Churn Modeling.csv')
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
print(df.shape)
print(df.info())
print(df.describe())

In [ ]:
df.isnull().sum()

,0
RowNumber,0
CustomerId,0
Surname,0
CreditScore,0
Geography,0
Gender,0
Age,0
Tenure,0
Balance,0
NumOfProducts,0


In [ ]:
df.duplicated().sum()

np.int64(0)

In [7]:
# Cleaning the column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[^\w_]", "", regex=True)
    .str.replace(r"_+", "_", regex=True)
)
df.columns

Index(['rownumber', 'customerid', 'surname', 'creditscore', 'geography',
       'gender', 'age', 'tenure', 'balance', 'numofproducts', 'hascrcard',
       'isactivemember', 'estimatedsalary', 'exited'],
      dtype='object')

In [8]:
y=df['exited']
X=df.drop(['exited'],axis=1)

In [9]:
X.columns

Index(['rownumber', 'customerid', 'surname', 'creditscore', 'geography',
       'gender', 'age', 'tenure', 'balance', 'numofproducts', 'hascrcard',
       'isactivemember', 'estimatedsalary'],
      dtype='object')

In [10]:
num_col=df.select_dtypes(include=np.number).columns
cat_col=df.select_dtypes(exclude=np.number).columns

skewness=df[num_col].skew()
print(skewness)

rownumber          0.000000
customerid         0.001149
creditscore       -0.071607
age                1.011320
tenure             0.010991
balance           -0.141109
numofproducts      0.745568
hascrcard         -0.901812
isactivemember    -0.060437
estimatedsalary    0.002085
exited             1.471611
dtype: float64


In [ ]:
#distrtibution of target variable

for i in num_col:
    plt.figure(figsize=(6,4))
    sns.histplot(df[i], kde=True)
    plt.title(f"Histogram of {i}")
    plt.show()
    plt.close()



In [ ]:

#outliers check
for i in df.columns:
    plt.figure(figsize=(6,4))
    sns.boxplot(df[i])
    plt.title(f"Boxplot of {i}")
    plt.show()
    plt.close()


In [11]:
# Dropping irrelevent columns
X.drop(columns=['rownumber','customerid','surname'],inplace=True)


In [ ]:
X['geography'].value_counts()

,count
geography,
France,5014
Germany,2509
Spain,2477


In [12]:
# Mapping cat cols
X['gender']=X['gender'].map({'Male':1,'Female':0})
X = pd.get_dummies(X, columns=['geography'], drop_first=True)


In [ ]:
X.head()

,creditscore,gender,age,tenure,balance,numofproducts,hascrcard,isactivemember,estimatedsalary,geography_Germany,geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,False,False
1,608,0,41,1,83807.86,1,0,1,112542.58,False,True
2,502,0,42,8,159660.80,3,1,0,113931.57,False,False
3,699,0,39,1,0.00,2,0,0,93826.63,False,False
4,850,0,43,2,125510.82,1,1,1,79084.10,False,True


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Step 1: Train and Temp split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Step 2: Validation and Test split
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

# Step 3: Feature Scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# model=Sequential()
# model.add(Dense(256,activation='relu',input_dim=11))
# Dropout(0.2)
# model.add(Dense(128,activation='relu'))
# Dropout(0.3)
# model.add(Dense(64,activation='relu'))
# Dropout(0.3)
# model.add(Dense(32,activation='relu'))
# Dropout(0.3)
# model.add(Dense(16,activation='relu'))
# Dropout(0.3)
# model.add(Dense(1,activation='sigmoid'))

In [27]:
from tensorflow.keras.regularizers import l2
model=Sequential()
model.add(Dense(64, activation='relu',input_dim=X_train_scaled.shape[1],kernel_regularizer=l2(0.001)))
model.add(Dropout(0.2))
model.add(Dense(32, activation='relu', kernel_regularizer=l2(0.001)))
model.add(Dropout(0.2))
model.add(Dense(16, activation='relu', kernel_regularizer=l2(0.001)))
model.add(Dropout(0.2))
model.add(Dense(8, activation='relu', kernel_regularizer=l2(0.001)))
model.add(Dense(1, activation='sigmoid'))

In [28]:
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

In [29]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_20 (Dense)                │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,521 (13.75 KB)

 Trainable params: 3,521 (13.75 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
from tensorflow.keras.callbacks import EarlyStopping
callback = EarlyStopping(
    monitor="val_loss",
    min_delta=0.001,
    patience=10,
    verbose=1,
    mode="auto"
)

In [31]:
history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=210,
    batch_size=32,
    verbose=1,
    callbacks=[callback]
)

Epoch 1/210
219/219 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.7923 - loss: 0.5700 - val_accuracy: 0.8107 - val_loss: 0.4912
Epoch 2/210
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7924 - loss: 0.5111 - val_accuracy: 0.8107 - val_loss: 0.4610
Epoch 3/210
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7984 - loss: 0.4875 - val_accuracy: 0.8393 - val_loss: 0.4364
Epoch 4/210
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8264 - loss: 0.4621 - val_accuracy: 0.8660 - val_loss: 0.4100
Epoch 5/210
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8379 - loss: 0.4417 - val_accuracy: 0.8727 - val_loss: 0.3853
Epoch 6/210
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8446 - loss: 0.4181 - val_accuracy: 0.8720 - val_loss: 0.3748
Epoch 7/210
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8436 - loss: 0.4183 - val_accuracy: 0.8707 - val_loss: 0.3759
Epoch 8/210
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8523 - loss: 0.4009 - val_accu

In [32]:
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test)

print("Test Accuracy:", test_accuracy)

47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8627 - loss: 0.3657
Test Accuracy: 0.862666666507721


In [33]:
y_pred=model.predict(X_test_scaled)

47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [34]:
from sklearn.metrics import r2_score
r2_score(y_test,y_pred)

0.34766530990600586

In [35]:
from sklearn.metrics import classification_report, confusion_matrix
y_pred = model.predict(X_test_scaled)
y_pred = (y_pred > 0.4).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
[[1098  102]
 [ 132  168]]
              precision    recall  f1-score   support

           0       0.89      0.92      0.90      1200
           1       0.62      0.56      0.59       300

    accuracy                           0.84      1500
   macro avg       0.76      0.74      0.75      1500
weighted avg       0.84      0.84      0.84      1500



In [ ]:
df['exited'].value_counts()

,count
exited,
0,7963
1,2037


In [142]:
class_weight = {0:1, 1:4}

history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=20,
    batch_size=32,
    class_weight=class_weight
)

Epoch 1/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.8637 - loss: 0.6444 - val_accuracy: 0.7780 - val_loss: 0.4752
Epoch 2/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8620 - loss: 0.6216 - val_accuracy: 0.7500 - val_loss: 0.5272
Epoch 3/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8789 - loss: 0.5800 - val_accuracy: 0.7780 - val_loss: 0.4972
Epoch 4/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8800 - loss: 0.5548 - val_accuracy: 0.7807 - val_loss: 0.4857
Epoch 5/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8861 - loss: 0.5169 - val_accuracy: 0.7880 - val_loss: 0.4922
Epoch 6/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8924 - loss: 0.4884 - val_accuracy: 0.7680 - val_loss: 0.5154
Epoch 7/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8969 - loss: 0.4791 - val_accuracy: 0.7873 - val_loss: 0.5045
Epoch 8/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9027 - loss: 0.4566 - val_accuracy: 0

In [37]:
model.save("churn_pred_model.keras")

In [36]:
## Hyper parameter tuning using keras tuner

!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 1.3 MB/s eta 0:00:00


In [38]:
import keras_tuner as kt
from tensorflow.keras.optimizers import Adam
def build_model(hp):
    model = Sequential()

    # First hidden layer
    model.add(Dense(
        units=hp.Int('units_1', min_value=16, max_value=64, step=16),
        activation='relu',
        kernel_regularizer=l2(hp.Float('l2_1', 1e-4, 1e-2, sampling='log')),
        input_dim=X_train_scaled.shape[1]
    ))

    model.add(Dropout(
        hp.Float('dropout_1', min_value=0.1, max_value=0.4, step=0.1)
    ))

    # Second hidden layer
    model.add(Dense(
        units=hp.Int('units_2', min_value=8, max_value=32, step=8),
        activation='relu',
        kernel_regularizer=l2(hp.Float('l2_2', 1e-4, 1e-2, sampling='log'))
    ))

    model.add(Dropout(
        hp.Float('dropout_2', min_value=0.1, max_value=0.4, step=0.1)
    ))

    # Output layer
    model.add(Dense(1, activation='sigmoid'))

    # Learning rate tuning
    lr = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [39]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=1,
    directory='keras_tuner_dir',
    project_name='customer_churn_tuning'
)

In [40]:
tuner.search(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=20,
    batch_size=32
)

Trial 10 Complete [00h 00m 19s]
val_accuracy: 0.871999979019165

Best val_accuracy So Far: 0.8733333349227905
Total elapsed time: 00h 03m 50s


In [41]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("Best units_1:", best_hp.get('units_1'))
print("Best dropout_1:", best_hp.get('dropout_1'))
print("Best units_2:", best_hp.get('units_2'))
print("Best dropout_2:", best_hp.get('dropout_2'))
print("Best learning rate:", best_hp.get('learning_rate'))

Best units_1: 16
Best dropout_1: 0.1
Best units_2: 32
Best dropout_2: 0.30000000000000004
Best learning rate: 0.001


In [42]:
best_model = tuner.hypermodel.build(best_hp)

In [43]:
history = best_model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=20,
    batch_size=32
)

Epoch 1/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7666 - loss: 0.5808 - val_accuracy: 0.8133 - val_loss: 0.4885
Epoch 2/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8069 - loss: 0.5016 - val_accuracy: 0.8260 - val_loss: 0.4518
Epoch 3/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8081 - loss: 0.4768 - val_accuracy: 0.8320 - val_loss: 0.4337
Epoch 4/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8186 - loss: 0.4579 - val_accuracy: 0.8420 - val_loss: 0.4132
Epoch 5/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8260 - loss: 0.4339 - val_accuracy: 0.8640 - val_loss: 0.3924
Epoch 6/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8389 - loss: 0.4201 - val_accuracy: 0.8727 - val_loss: 0.3721
Epoch 7/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8393 - loss: 0.4048 - val_accuracy: 0.8680 - val_loss: 0.3653
Epoch 8/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8450 - loss: 0.3970 - val_accuracy: 0.

In [44]:
loss, accuracy = best_model.evaluate(X_test_scaled, y_test)
print("Test Accuracy:", accuracy)

47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8660 - loss: 0.3534
Test Accuracy: 0.8659999966621399


In [45]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = (best_model.predict(X_test_scaled) > 0.5).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step
[[1156   44]
 [ 157  143]]
              precision    recall  f1-score   support

           0       0.88      0.96      0.92      1200
           1       0.76      0.48      0.59       300

    accuracy                           0.87      1500
   macro avg       0.82      0.72      0.75      1500
weighted avg       0.86      0.87      0.85      1500



In [46]:
# =========================
# CUSTOMER INPUTS
# =========================

creditscore = float(input("Enter Credit Score: "))
gender = input("Enter Gender (Male/Female): ")
age = float(input("Enter Age: "))
tenure = float(input("Enter Tenure: "))
balance = float(input("Enter Balance: "))
numofproducts = float(input("Enter Number of Products: "))
hascrcard = int(input("Has Credit Card? (1/0): "))
isactivemember = int(input("Is Active Member? (1/0): "))
estimatedsalary = float(input("Enter Estimated Salary: "))
geography = input("Enter Geography (France/Germany/Spain): ")


Enter Credit Score: 850
Enter Gender (Male/Female): F
Enter Age: 41
Enter Tenure: 6
Enter Balance: 160000
Enter Number of Products: 3
Has Credit Card? (1/0): 1
Is Active Member? (1/0): 1
Enter Estimated Salary: 120000
Enter Geography (France/Germany/Spain): France


In [47]:
# =========================
# ENCODING INPUTS
# =========================

# Gender encoding
if gender == 'Male':
    gender = 1
else:
    gender = 0

# Geography encoding
if geography == 'Germany':
    geography_Germany = 1
    geography_Spain = 0
elif geography == 'Spain':
    geography_Germany = 0
    geography_Spain = 1
else:   # France
    geography_Germany = 0
    geography_Spain = 0

# =========================
# CREATE INPUT ARRAY
# SAME ORDER AS TRAINING
# =========================

user_data = np.array([[
    creditscore,
    gender,
    age,
    tenure,
    balance,
    numofproducts,
    hascrcard,
    isactivemember,
    estimatedsalary,
    geography_Germany,
    geography_Spain
]])

# =========================
# SCALE INPUT
# =========================

user_data_scaled = scaler.transform(user_data)

# =========================
# PREDICT
# =========================

prediction = model.predict(user_data_scaled)

churn_probability = prediction[0][0]

print("\nChurn Probability:", churn_probability)

# =========================
# FINAL DECISION
# =========================

if churn_probability > 0.5:
    print("Prediction: Customer is likely to CHURN")
else:
    print("Prediction: Customer is NOT likely to churn")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step

Churn Probability: 0.9297905
Prediction: Customer is likely to CHURN
